import libraries

In [2]:
import os
import json
import pandas as pd
import numpy as np
from PIL import Image

 for our TARGET DATASET that is quadrant ennumeration disease making a path address. 

In [3]:
data_path=os.path.join("training_data","quadrant-enumeration-disease")
json_file_path=os.path.join(data_path,"train_quadrant_enumeration_disease.json")



now opening these files

In [4]:
with open(json_file_path,"r") as f:
    json_data=json.load(f)
print("json loaded successfully")

json loaded successfully


Checking json data has:

In [5]:
print(json_data.keys())

dict_keys(['images', 'annotations', 'categories_1', 'categories_2', 'categories_3'])


In [6]:
images=json_data["images"]
annotations=json_data["annotations"]
print(len(images))
print(f"first value of images in the list={images[0]}")
print(len(annotations))
print(f"first value of annnotaions in the list={annotations[0]}")

705
first value of images in the list={'height': 1316, 'width': 2744, 'id': 1, 'file_name': 'train_673.png'}
3529
first value of annnotaions in the list={'iscrowd': 0, 'image_id': 1, 'bbox': [542.0, 698.0, 220.0, 271.0], 'segmentation': [[621, 703, 573, 744, 542, 885, 580, 945, 650, 969, 711, 883, 762, 807, 748, 741, 649, 698]], 'id': 1, 'area': 39683, 'category_id_1': 3, 'category_id_2': 7, 'category_id_3': 0}


In [7]:
print(json_data["categories_1"])
quadrant={}
for c in json_data["categories_1"]:
    quadrant[c["id"]]=c["name"]
print(f"quadrant={quadrant}")

print(json_data["categories_2"])
tooth_num={}
for t in json_data["categories_2"]:
    tooth_num[t["id"]]=t["name"]
print(f"tooth numbers={tooth_num}")

print(json_data["categories_3"])
disease={}
for d in json_data["categories_3"]:
    disease[d["id"]]=d["name"]
print(f"disease={disease}")


[{'id': 0, 'name': '1', 'supercategory': '1'}, {'id': 1, 'name': '2', 'supercategory': '2'}, {'id': 2, 'name': '3', 'supercategory': '3'}, {'id': 3, 'name': '4', 'supercategory': '4'}]
quadrant={0: '1', 1: '2', 2: '3', 3: '4'}
[{'id': 0, 'name': '1', 'supercategory': '1'}, {'id': 1, 'name': '2', 'supercategory': '2'}, {'id': 2, 'name': '3', 'supercategory': '3'}, {'id': 3, 'name': '4', 'supercategory': '4'}, {'id': 4, 'name': '5', 'supercategory': '5'}, {'id': 5, 'name': '6', 'supercategory': '6'}, {'id': 6, 'name': '7', 'supercategory': '7'}, {'id': 7, 'name': '8', 'supercategory': '8'}]
tooth numbers={0: '1', 1: '2', 2: '3', 3: '4', 4: '5', 5: '6', 6: '7', 7: '8'}
[{'id': 0, 'name': 'Impacted', 'supercategory': 'Impacted'}, {'id': 1, 'name': 'Caries', 'supercategory': 'Caries'}, {'id': 2, 'name': 'Periapical Lesion', 'supercategory': 'Periapical Lesion'}, {'id': 3, 'name': 'Deep Caries', 'supercategory': 'Deep Caries'}]
disease={0: 'Impacted', 1: 'Caries', 2: 'Periapical Lesion', 3: 

To crop images we have 2 requirement :
from images list we need image id and file name to be put in a dictionary

In [8]:
image_find={}
for i in images:
    image_find[i["id"]]=i["file_name"]
print(len(image_find))

705


In order to crop images we need three 3 things from annotation:
1 - image id 
2- bbox
3- categories3 and its id, 


cropping :
1. taking annotation image_id for file name(part 2 of annotation)
 
2. then we will connect this with image find ,from that file_name opening that image using pil image
3. annotation bbox used to crop(part 3 of annotation)
4. and saving crop image in disease folder based on disease type.

testing single photo crop

In [9]:
original_img_path= os.path.join(data_path,"xrays")
# extracting id and extracting them together
ann = annotations[0]
imgid= ann["image_id"]
file =image_find[imgid]
imgpath =os.path.join(original_img_path, file)
i_one= Image.open(imgpath)

# bounding box coordinates
x,y,w,h= ann["bbox"]
left=x
top=y
right= w+x
bottom=y+h

# crop
crop_1= i_one.crop((left,top,right,bottom))
crop_1.show()


now make new folder for all the data based on disease to store cropped images

In [10]:
import shutil
shutil.rmtree("output", ignore_errors=True)

In [11]:
output_base="output"
for d in disease:
    disease_name=disease[d]
    folder_path=os.path.join(output_base, disease_name)
    os.makedirs(folder_path,exist_ok=True)

cropping all the xray images

In [12]:
# doing part 2 of annotation to crop photos for all the images
# taking image id from annootaion

for ann in annotations:
    i_id=ann["image_id"]
    file_name=image_find[i_id]
    path=os.path.join(original_img_path,file_name)
    i_all=Image.open(path)

# bounding box part

    x,y,w,h=ann["bbox"]
    left=x
    top=y
    right=x+w
    bottom=y+h
    crop_img=i_all.crop((left,top,right,bottom))
    
# saving crop images into right folder

    d_id=ann["category_id_3"]
    disease_type=disease[d_id]
    file_naming= f"croptooth_{ann['id']}.png"
    save_path=os.path.join(output_base,disease_type,file_naming)
    crop_img.save(save_path)
    


In [13]:
#  checking length of all the folder 
for d in disease:
    disease_name=disease[d]
    folder_path=os.path.join(output_base,disease_name)
    files=os.listdir(folder_path)
    print(f"{disease_name} images count: {len(files)} ")

Impacted images count: 604 
Caries images count: 2189 
Periapical Lesion images count: 158 
Deep Caries images count: 578 


Now our data is cropped and we will do train test split on this and then move data into new test/ disease_type folder and validation disease_type folder 

In [14]:
import sklearn
import shutil
from sklearn.model_selection import train_test_split
train_val_data = "dataset"
# splitting data based on disease
for d in disease:
   disease_folder = disease[d]
   folder=os.path.join(output_base,disease_folder)
   files=os.listdir(folder)
   train_files,val_files = train_test_split(files, test_size=0.2, random_state=42)
   
   #  making new folder destination for train data and val data
   train_path = os.path.join(train_val_data,"train",disease_folder)
   val_path = os.path.join(train_val_data,"val",disease_folder)
   os.makedirs(train_path, exist_ok=True)
   os.makedirs(val_path,exist_ok=True)  
    #  copying the data into the respective folder
   for f in train_files:
       source = os.path.join(folder,f)
       destination = os.path.join(train_path,f)
       shutil.copy(source,destination)
   for f in val_files:
       source = os.path.join(folder, f)
       destination = os.path.join(val_path,f)
       shutil.copy(source, destination)
   train_count=(len(os.listdir(train_path)))
   val_count= len(os.listdir(val_path))
   print(f"{disease_folder} ,train= {train_count} ;val={val_count}")
   

Impacted ,train= 483 ;val=121
Caries ,train= 1751 ;val=438
Periapical Lesion ,train= 126 ;val=32
Deep Caries ,train= 462 ;val=116


 json_data       → poora JSON file, dictionary format mein (images, annotations, categories_1/2/3)

images          → list, saari X-ray images ki info (id, filename, height, width)

annotations     → list, saare 3529 tooth-markings (bbox, image_id, category ids)

quadrant        → dict, {0:'1', 1:'2', 2:'3', 3:'4'} — quadrant id → naam

 tooth_num       → dict, {0:'1', ..., 7:'8'} — tooth id → naam

 disease         → dict, {0:'Impacted', 1:'Caries', 2:'Periapical Lesion', 3:'Deep Caries'} — disease id → naam


 image_find      → dict, {image_id: filename} — fast lookup ke liye

img_folder      → string, path jaha saari original X-ray images hain

output_base     → string "output" — jaha cropped, disease-sorted images save hui

train_val_data  → string "dataset" — jaha train/val split ke baad images gayi